In [ ]:
# 1. check that a GPU is attached
!nvidia-smi

: 

In [ ]:
# 2. install
!pip install -q transformers pycocoevalcap

In [ ]:
# 3. connect Google Drive and set the output folder
from google.colab import drive
drive.mount('/content/drive')

import os
OUT_DIR = "/content/drive/MyDrive/FYP/blip_vizwiz"
os.makedirs(OUT_DIR, exist_ok=True)
print("results go to", OUT_DIR)

In [ ]:
# 4. download VizWiz (captions + train and val images) into Colab's local disk
#    the image zips are large, so this can take a while. Test images are skipped
#    because their captions are hidden.
%cd /content
!mkdir -p vizwiz
!wget -q --show-progress -nc -P vizwiz https://vizwiz.cs.colorado.edu/VizWiz_final/caption/annotations.zip
!wget -q --show-progress -nc -P vizwiz https://vizwiz.cs.colorado.edu/VizWiz_final/images/train.zip
!wget -q --show-progress -nc -P vizwiz https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip
!unzip -q -o vizwiz/annotations.zip -d vizwiz/ann
!unzip -q -o vizwiz/train.zip -d vizwiz/img
!unzip -q -o vizwiz/val.zip -d vizwiz/img
!ls vizwiz/ann vizwiz/img

In [ ]:
# 5. read the captions and drop the bad ones
#    is_rejected = spam caption
#    is_precanned = the stock "Quality issues are too severe..." text
#    images whose captions are all bad are dropped completely
import json, random, re
from collections import defaultdict

random.seed(0)
ROOT = "/content/vizwiz"

img_path = {}
for d, _, files in os.walk(os.path.join(ROOT, "img")):
    for f in files:
        if f.endswith(".jpg"):
            img_path[f] = os.path.join(d, f)
print("images on disk:", len(img_path))

def find_file(name):
    for d, _, files in os.walk(os.path.join(ROOT, "ann")):
        if name in files:
            return os.path.join(d, name)
    raise FileNotFoundError(name)

def load_split(split):
    data = json.load(open(find_file(split + ".json")))
    names = {im["id"]: im["file_name"] for im in data["images"]}
    caps = defaultdict(list)
    dropped = 0
    for a in data["annotations"]:
        if a.get("is_rejected") or a.get("is_precanned"):
            dropped += 1
            continue
        c = a["caption"].replace("\r", " ").strip()   # old files had a stray \r
        if c:
            caps[names[a["image_id"]]].append(c)
    items = [(n, c) for n, c in caps.items() if n in img_path]
    print(split, "- images kept:", len(items), "| captions dropped:", dropped)
    return items

train_all = load_split("train")
val_all = load_split("val")
print("example:", train_all[0])

In [ ]:
# 6. choose how much data to use
#    VizWiz test captions are hidden, so the official val split is cut into
#    dev (checked after every epoch) and test (only used for the final numbers).
N_TRAIN = 8000    # start here; raise to len(train_all) once everything works
N_DEV   = 500
N_TEST  = 1000

random.shuffle(train_all)
random.shuffle(val_all)
train_items = train_all[:N_TRAIN]
dev_items   = val_all[:N_DEV]
test_items  = val_all[N_DEV:N_DEV + N_TEST]

json.dump({"train": [n for n, _ in train_items],
           "dev":   [n for n, _ in dev_items],
           "test":  [n for n, _ in test_items]},
          open(os.path.join(OUT_DIR, "split.json"), "w"))
print("train", len(train_items), "| dev", len(dev_items), "| test", len(test_items))

In [ ]:
# 7. helpers: caption a list of images, and score captions with BLEU-4 and CIDEr
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider

MODEL_NAME = "Salesforce/blip-image-captioning-base"
device = "cuda"
processor = BlipProcessor.from_pretrained(MODEL_NAME)

@torch.no_grad()
def caption_all(model, names, bs=32):
    model.eval()
    out = []
    for i in range(0, len(names), bs):
        imgs = [Image.open(p).convert("RGB") for p in names[i:i + bs]]
        pv = processor(images=imgs, return_tensors="pt").pixel_values.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            ids = model.generate(pixel_values=pv, num_beams=3, max_new_tokens=30)
        out += processor.batch_decode(ids, skip_special_tokens=True)
    return out

def norm(s):
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", s.lower()).split())

def score(preds, items):
    # simple lowercase + no punctuation, so no Java tokenizer is needed
    gts = {i: [norm(c) for c in caps] for i, (_, caps) in enumerate(items)}
    res = {i: [norm(preds[i])] for i in range(len(items))}
    bleu, _ = Bleu(4).compute_score(gts, res)
    cider, _ = Cider().compute_score(gts, res)
    return {"BLEU-4": round(bleu[3], 3), "CIDEr": round(cider, 3)}

def paths(items):
    return [img_path[n] for n, _ in items]

In [ ]:
# 8. BASELINE: the original BLIP-base on the test set, before any training
base = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
base_preds = caption_all(base, paths(test_items))
base_scores = score(base_preds, test_items)
print("original BLIP-base on VizWiz test:", base_scores)

json.dump({"scores": base_scores, "preds": base_preds},
          open(os.path.join(OUT_DIR, "baseline_test.json"), "w"))
del base
torch.cuda.empty_cache()

In [ ]:
# 9. data loader: each epoch uses one random caption (out of ~5) per image
from torch.utils.data import Dataset, DataLoader

class CaptionData(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        name, caps = self.items[i]
        return Image.open(img_path[name]).convert("RGB"), random.choice(caps)

def collate(batch):
    imgs, caps = zip(*batch)
    enc = processor(images=list(imgs), text=list(caps), padding="max_length",
                    truncation=True, max_length=40, return_tensors="pt")
    labels = enc.input_ids.clone()
    labels[enc.attention_mask == 0] = -100    # don't learn from padding
    enc["labels"] = labels
    return enc

In [ ]:
# 10. fine-tune
#     The image encoder is frozen: faster, fits the T4 easily, and the model
#     forgets less of what it knew about normal scenes. Only the text decoder learns.
from transformers import get_cosine_schedule_with_warmup

EPOCHS = 3
BATCH  = 16
LR     = 3e-5

model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
for p in model.vision_model.parameters():
    p.requires_grad = False
bos = model.config.text_config.bos_token_id   # generate() starts with this token, so training must too

loader = DataLoader(CaptionData(train_items), batch_size=BATCH, shuffle=True,
                    num_workers=2, collate_fn=collate)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=LR, weight_decay=0.05)
total_steps = EPOCHS * len(loader)
sched = get_cosine_schedule_with_warmup(opt, int(0.05 * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda")

best_cider = -1
history = []
for ep in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for step, b in enumerate(loader, 1):
        b = {k: v.to(device) for k, v in b.items()}
        b["input_ids"][:, 0] = bos
        b["labels"][:, 0] = bos
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(pixel_values=b["pixel_values"], input_ids=b["input_ids"],
                         attention_mask=b["attention_mask"], labels=b["labels"]).loss
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        running += loss.item()
        if step % 100 == 0:
            print(f"epoch {ep} step {step}/{len(loader)} loss {running / 100:.3f}")
            running = 0.0

    dev_scores = score(caption_all(model, paths(dev_items)), dev_items)
    history.append({"epoch": ep, **dev_scores})
    print(f"epoch {ep} dev:", dev_scores)

    model.save_pretrained(os.path.join(OUT_DIR, "last"))
    processor.save_pretrained(os.path.join(OUT_DIR, "last"))
    if dev_scores["CIDEr"] > best_cider:
        best_cider = dev_scores["CIDEr"]
        model.save_pretrained(os.path.join(OUT_DIR, "best"))
        processor.save_pretrained(os.path.join(OUT_DIR, "best"))
        print("  saved as best")

json.dump(history, open(os.path.join(OUT_DIR, "history.json"), "w"))

In [ ]:
# 11. FINAL: fine-tuned model on the same test set, compared with the baseline
best = BlipForConditionalGeneration.from_pretrained(os.path.join(OUT_DIR, "best")).to(device)
ft_preds = caption_all(best, paths(test_items))
ft_scores = score(ft_preds, test_items)
base_saved = json.load(open(os.path.join(OUT_DIR, "baseline_test.json")))

print("              BLEU-4   CIDEr")
print("original     ", base_saved["scores"]["BLEU-4"], "  ", base_saved["scores"]["CIDEr"])
print("fine-tuned   ", ft_scores["BLEU-4"], "  ", ft_scores["CIDEr"])
json.dump({"scores": ft_scores, "preds": ft_preds},
          open(os.path.join(OUT_DIR, "finetuned_test.json"), "w"))

print()
for i in random.sample(range(len(test_items)), 10):
    print(test_items[i][0])
    print("  original  :", base_saved["preds"][i])
    print("  fine-tuned:", ft_preds[i])
    print("  human     :", test_items[i][1][0])

In [ ]:
# 12. OPTIONAL: compare both models on your own photos
#     upload your test photos to Drive in FYP/test_images first
own_dir = "/content/drive/MyDrive/FYP/test_images"
own = sorted(os.path.join(own_dir, f) for f in os.listdir(own_dir)
             if f.lower().endswith((".jpg", ".jpeg", ".png")))
base = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
for p, a, b in zip(own, caption_all(base, own), caption_all(best, own)):
    print(os.path.basename(p))
    print("  original  :", a)
    print("  fine-tuned:", b)

To use the fine-tuned model in your pipeline, download the `FYP/blip_vizwiz/best` folder and load it with
`BlipForConditionalGeneration.from_pretrained("path/to/best")` instead of the Salesforce name.